In [16]:
import matplotlib.pyplot as plt
import numpy as np

import tidy3d as td

import gplugins as gp
import gplugins.tidy3d as gt
from gplugins import plot
from gplugins.common.config import PATH

import gplugins.tidy3d.materials as mat

nm = 1e-3
wavelength = np.linspace(1500, 1600) * nm
f = td.C_0 / wavelength

In [5]:
nitride_complex = td.material_library["Si3N4"]["Luke2015PMLStable"].eps_model(f)
nitride_index, nitride_k = td.Medium.eps_complex_to_nk(nitride_complex)
box_complex = td.material_library["SiO2"]["Horiba"].eps_model(f)
box_index, box_k = td.Medium.eps_complex_to_nk(box_complex)

## 1. Simulating Propagation Modes in SOI Waveguides

In [4]:
# STUDENT! Put your code here.
wavelength = np.linspace(1500,1600,6) * nm  # Student code here. Tip np.linspace()
central = 1550*nm

deep_waveguide = gt.modes.Waveguide(
    wavelength=wavelength, 
    core_width=450 * nm, 
    slab_thickness=0.0,
    core_material='si',
    clad_material='sio2',
    core_thickness=220 * nm,
    num_modes=8,
    cache_path='.cache/',
    precision='double',
    max_grid_scaling=1.2,
    grid_resolution=20, 
)

res_neff = deep_waveguide.n_eff # In this case, the result is not just a number, is a wavelength-dependent vector
res_te = deep_waveguide.fraction_te # Wavelength-dependent vector
res_tm =deep_waveguide.fraction_tm # Wavelength-dependent vector
neff_TE0 = res_neff[:,0].real

coef_TE = np.polyfit(wavelength - central,neff_TE0,2)

n1_TE = coef_TE[2]
n2_TE = coef_TE[1]
n3_TE = coef_TE[0]

ng_TE = n1_TE - central*n2_TE
neff_TM0 = res_neff[:,1].real

coef_TM = np.polyfit(wavelength- central,neff_TM0,2)

n1_TM = coef_TM[2]
n2_TM = coef_TM[1]
n3_TM = coef_TM[0]

ng_TM = n1_TM - central*n2_TM
print(
    f"""
TE0 effective index varies from {neff_TE0[0]:.3f} to {neff_TE0[-1]:.3f}
TM0 effective index varies from {neff_TM0[0]:.3f} to {neff_TM0[-1]:.3f}

ng_TE = {ng_TE:.3f}
ng_TM = {ng_TM:.3f}
"""
)

2026-03-23 10:09:57.541 | INFO     | gplugins.tidy3d.modes:_data:266 - load data from .cache\Waveguide_e0ec38b5ceb294b8.npz.

TE0 effective index varies from 2.399 to 2.273
TM0 effective index varies from 1.806 to 1.688

ng_TE = 4.287
ng_TM = 3.571



## 2. Designing Directional Couplers

Primero para un rango de valores de gap y otro de L, calculamos los valores de acoplo y luego de la matriz resultante filtramos los valores de acoplo que necesitamos para ver con que valores de gap y de L se consiguen.

In [31]:
lambda_c = 1.55  # wavelength (um)

gap = np.linspace(0.6, 1, 10)  # gap sweep (um)

K_vals = []
Lpi = []

for g in gap:

    dcoupler_cs = gt.modes.WaveguideCoupler(
        # Geometrical Parameters
        core_width=(1.2, 1.2),      # waveguide widths
        slab_thickness=0 * nm,
        core_thickness=300 * nm,
        gap=g,                      # swept parameter
        # Materials
        core_material="si",
        clad_material="sio2",
        # Modesolver Parameters
        wavelength=lambda_c,
        num_modes=5,
        cache_path=".cache/",
        precision="double",
        max_grid_scaling=1.5,
        grid_resolution=20,
    )

    # calcular los modos
    modes = dcoupler_cs.n_eff

   # índices efectivos de los dos supermodos
    neff_even = modes[0].real
    neff_odd = modes[4].real

    # longitud de acoplo π
    L_pi = lambda_c / (2 * (neff_even - neff_odd))
    Lpi.append(L_pi)


Lpi = np.array(Lpi)

# barrido de longitud del acoplador
L = np.linspace(0, 8 * np.max(Lpi), 200)

for L_pi in Lpi:
    K = np.sin(np.pi / 2 * (L / L_pi)) ** 2
    K_vals.append(K)

K_vals = np.array(K_vals)

2026-03-23 10:56:37.130 | INFO     | gplugins.tidy3d.modes:_data:266 - load data from .cache\WaveguideCoupler_339095014771a6d9.npz.
2026-03-23 10:56:37.136 | INFO     | gplugins.tidy3d.modes:_data:266 - load data from .cache\WaveguideCoupler_6e56805929e7cd39.npz.
2026-03-23 10:56:37.141 | INFO     | gplugins.tidy3d.modes:_data:266 - load data from .cache\WaveguideCoupler_a3e0f0a1cd79c0d6.npz.
2026-03-23 10:56:37.148 | INFO     | gplugins.tidy3d.modes:_data:266 - load data from .cache\WaveguideCoupler_2441b20c6cc88d56.npz.
2026-03-23 10:56:37.154 | INFO     | gplugins.tidy3d.modes:_data:266 - load data from .cache\WaveguideCoupler_64ae397af35025ff.npz.
2026-03-23 10:56:37.160 | INFO     | gplugins.tidy3d.modes:_data:266 - load data from .cache\WaveguideCoupler_cf45c78ddfd83ae6.npz.
2026-03-23 10:56:37.169 | INFO     | gplugins.tidy3d.modes:_data:266 - load data from .cache\WaveguideCoupler_b4b5ff7fe5ba6bb2.npz.
2026-03-23 10:56:37.176 | INFO     | gplugins.tidy3d.modes:_data:266 - load 

In [32]:
k_objetivos = [0.5, 0.3, 0.23, 0.17, 0.05]

for k in k_objetivos:

    diff = np.abs(K_vals - k)

    # índice del valor más cercano
    idx = np.unravel_index(np.argmin(diff), diff.shape)

    i, j = idx

    print(f"K objetivo = {k}")
    print(f"gap = {gap[i]:.4f}, L = {L[j]:.4f}, K = {K_vals[i,j]:.4f}\n")

K objetivo = 0.5
gap = 0.8222, L = 6.4140, K = 0.5038

K objetivo = 0.3
gap = 0.8222, L = 10.3215, K = 0.3039

K objetivo = 0.23
gap = 0.8222, L = 14.0814, K = 0.2340

K objetivo = 0.17
gap = 0.8222, L = 3.1702, K = 0.1709

K objetivo = 0.05
gap = 0.8222, L = 7.5936, K = 0.0481

